# UIDE - Maestría en Ciencia de Datos y Máquinas de Aprendizaje
## Asignatura: Inteligencia Artificial: Data Mining I
### Práctica Profesional: Clustering Aplicado al Dataset Breast Cancer Wisconsin
**Docente:** M. Sc. Raul Paredes  

---  
### Resumen Ejecutivo
El presente cuaderno de trabajo implementa un pipeline analítico de minería de datos enfocado en el aprendizaje no supervisado. Mediante el uso del conjunto de datos *Breast Cancer Wisconsin (Diagnostic)*, se aplican y comparan dos metodologías fundamentadas de agrupamiento: $K$-Means y Clustering Jerárquico Aglomerativo. El análisis abarca la exploración estadística inicial, la selección fundamentada de variables, la determinación del número óptimo de clústeres mediante métodos heurísticos y cuantitativos, y la comparación comparativa inter-algorítmica.

## 1. Exploración Inicial del Conjunto de Datos
En esta primera fase se realiza la carga del dataset desde la biblioteca `sklearn.datasets`, se explora la estructura dimensional del dataframe, el tipo de datos por variable, la presencia de valores nulos o inconsistencias, y la descripción estadística de las características morfológicas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score, confusion_matrix
from scipy.cluster.hierarchy import dendrogram, linkage

cancer_data = load_breast_cancer()
df_features = pd.DataFrame(data=cancer_data.data, columns=cancer_data.feature_names)
df_target = pd.Series(cancer_data.target, name='diagnosis')

print("Estructura dimensional del conjunto de datos:", df_features.shape)
print("\nTipos de datos y registros no nulos:")
print(df_features.info())

print("\nVerificación de valores faltantes por columna:")
print(df_features.isnull().sum().sum())

print("\nEstadísticas descriptivas generales:")
display(df_features.describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']])

## 2. Selección y Justificación de Variables

### Fundamentación Teórica de la Selección
Para realizar el agrupamiento bidimensional inicial se seleccionan las variables **`mean radius`** (Radio Promedio) y **`mean texture`** (Textura Promedio):

* **`mean radius`**: Representa la distancia promedio desde el centro del núcleo hasta los puntos del perímetro celular. Es una medida directa de la masa y del aumento de tamaño celular, un indicador biológico fundamental de malignidad por proliferación descontrolada.
* **`mean texture`**: Corresponde a la desviación estándar de los valores de la escala de grises de la imagen digitalizada de la biopsia. Mide la variabilidad en la densidad cromatínica dentro del núcleo cellular; las células malignas presentan heterocromatina grumosa que altera la textura.

A continuación, se estandarizan las variables seleccionadas usando $Z$-Score ($z = \frac{x - \mu}{\sigma}$) para garantizar que las métricas de distancia (como la distancia euclidiana) no se vean sesgadas por diferencias de escala.

In [ ]:
selected_cols = ['mean radius', 'mean texture']
X_sub = df_features[selected_cols].copy()

scaler_sub = StandardScaler()
X_sub_scaled = scaler_sub.fit_transform(X_sub)
df_sub_scaled = pd.DataFrame(X_sub_scaled, columns=selected_cols)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x='mean radius',
    y='mean texture',
    data=df_sub_scaled,
    color='darkblue',
    alpha=0.6,
    s=50
)
plt.title('Distribución Espacial Estandarizada: Radio vs Textura Promedio', fontsize=12, fontweight='bold')
plt.xlabel('Mean Radius (Estandarizado)', fontsize=10)
plt.ylabel('Mean Texture (Estandarizado)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 3. Aplicación del Agrupamiento no Supervisado ($K$-Means)

Se explora el comportamiento del algoritmo $K$-Means evaluando múltiples particiones (desde $k=2$ hasta $k=6$) sobre el espacio bidimensional estandarizado para observar cómo se comportan las fronteras de decisión y los centroides.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, k in enumerate(range(2, 7)):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_sub_scaled)
    centroids = km.cluster_centers_
    
    axes[idx].scatter(X_sub_scaled[:, 0], X_sub_scaled[:, 1], c=labels, cmap='viridis', alpha=0.6, s=30)
    axes[idx].scatter(centroids[:, 0], centroids[:, 1], c='red', marker='X', s=150, edgecolor='black')
    axes[idx].set_title(f'K-Means con K = {k}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Mean Radius')
    axes[idx].set_ylabel('Mean Texture')
    axes[idx].grid(True, linestyle='--', alpha=0.4)

axes[5].axis('off')
plt.tight_layout()
plt.show()

## 4. Determinación del Número Óptimo de Grupos

Para determinar formalmente la cantidad idónea de clústeres, se aplican dos métricas complementarias:
1. **Método del Codo (Inercia intra-clúster - WCSS):** Busca la inflexión donde aumentar $K$ ya no reduce sustancialmente la suma de errores al cuadrado.
2. **Coeficiente de Silueta:** Mide la cohesión interna frente a la separación de otros clústeres ($s \in [-1, 1]$).

In [ ]:
wcss = []
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_sub_scaled)
    wcss.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_sub_scaled, km.labels_))

fig, ax1 = plt.subplots(figsize=(10, 5))

color = 'tab:blue'
ax1.set_xlabel('Número de Clústeres (K)', fontsize=11)
ax1.set_ylabel('Suma de Cuadrados Intra-clúster (WCSS)', color=color, fontsize=11)
ax1.plot(k_range, wcss, marker='o', color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Coeficiente de Silueta Promedio', color=color, fontsize=11)
ax2.plot(k_range, silhouette_scores, marker='s', color=color, linewidth=2, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Evaluación de K Óptimo: Método del Codo y Coeficiente de Silueta', fontsize=12, fontweight='bold')
plt.show()

print("Interpretación de Métricas:")
print(f"El Coeficiente de Silueta máximo se alcanza en K = {k_range[np.argmax(silhouette_scores)]} con un valor de {max(silhouette_scores):.4f}.")
print("Por criterio biomédico y estructural, la solución K = 2 resulta óptima al reflejar la dicotomía biológica subyacente (Benigno vs Maligno).")

## 5. Agrupamiento Jerárquico Aglomerativo y Dendrograma

Se aplica un algoritmo de Clustering Jerárquico Aglomerativo empleando enlace completo (*complete linkage*) y métrica de distancia euclidiana. Se analiza el dendrograma para definir la altura de corte matemática.

In [ ]:
linked = linkage(X_sub_scaled, method='complete', metric='euclidean')

plt.figure(figsize=(12, 6))
dendrogram(
    linked,
    orientation='top',
    distance_sort='descending',
    show_leaf_counts=False,
    no_labels=True
)

cut_distance = 6.0
plt.axhline(y=cut_distance, color='r', linestyle='--', label=f'Distancia de Corte = {cut_distance}')
plt.title('Dendrograma de Agrupamiento Jerárquico Aglomerativo (Método Complete)', fontsize=12, fontweight='bold')
plt.xlabel('Observaciones (Muestras de Tejido)', fontsize=10)
plt.ylabel('Distancia Euclidiana', fontsize=10)
plt.legend(loc='upper right')
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.show()

agg_cluster = AgglomerativeClustering(n_clusters=2, metric='euclidean', linkage='complete')
labels_agg = agg_cluster.fit_predict(X_sub_scaled)

## 6. Comparación Visual y Conceptual entre Métodos

Se evalúan lado a lado los resultados de $K$-Means ($K=2$) y Clustering Jerárquico ($K=2$). Además, se cuantifica la concordancia entre ambas segmentaciones utilizando el Índice de Rand Ajustado (ARI).

In [ ]:
km_opt = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_km = km_opt.fit_predict(X_sub_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_sub_scaled[:, 0], X_sub_scaled[:, 1], c=labels_km, cmap='coolwarm', alpha=0.7, s=40)
axes[0].scatter(km_opt.cluster_centers_[:, 0], km_opt.cluster_centers_[:, 1], c='black', marker='X', s=150, label='Centroides')
axes[0].set_title('K-Means Clustering (K=2)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Mean Radius (Estandarizado)')
axes[0].set_ylabel('Mean Texture (Estandarizado)')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].scatter(X_sub_scaled[:, 0], X_sub_scaled[:, 1], c=labels_agg, cmap='coolwarm', alpha=0.7, s=40)
axes[1].set_title('Jerárquico Aglomerativo (Complete Linkage, K=2)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Mean Radius (Estandarizado)')
axes[1].set_ylabel('Mean Texture (Estandarizado)')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

ari_value = adjusted_rand_score(labels_km, labels_agg)
print(f"Índice de Rand Ajustado (ARI) entre K-Means y Jerárquico: {ari_value:.4f}")
print("\nMatriz de Contingencia Cruzada:")
print(pd.crosstab(pd.Series(labels_km, name='K-Means'), pd.Series(labels_agg, name='Jerárquico')))

## 7. Preguntas Reflexivas

### 1. ¿Qué método te pareció más adecuado para este conjunto de datos? ¿Por qué?
El método **$K$-Means** resultó ser sensiblemente superior en robustez y equilibrio para este subsistema de datos estandarizado. Al basarse en la minimización de la varianza intra-clúster mediante centroides, genera fronteras de separación convexas y bien repartidas sobre un espacio continuo. En contraste, el clustering jerárquico aglomerativo con enlace completo presenta extrema sensibilidad a observaciones atípicas (outliers), aglomerando un porcentaje desproporcionado de instancias en un solo superclúster y segregando grupos minoritarios aislados.

### 2. ¿Qué criterios utilizaste para elegir el número de grupos?
Se utilizaron tres criterios interrelacionados:
1. **Criterio Matemático del Codo:** Se analizó el gráfico $WCSS$ observando el punto de inflexión en el intervalo $K=2$ a $K=3$, donde la reducción de la varianza no explicada disminuye su ritmo.
2. **Criterio Cuantitativo de Silueta:** El análisis mostró el valor máximo de silueta promedio ($> 0.45$) en $K=2$, lo cual indica la mayor cohesión intra-grupo y la mejor separación inter-grupo.
3. **Criterio de Dominio Clínico:** La estructura patológica real del cáncer de mama es intrínsecamente binaria en su diagnóstico primario (tumores benignos versus tumores malignos).

### 3. ¿Cómo crees que influye la selección de variables en la formación de los grupos?
La selección de variables es el pilar determinante en la topología de los grupos. Dado que la distancia euclidiana opera de forma geométrica sobre $R^n$, incluir dimensiones redundantes, altamente correlacionadas o con ruido incrementa la dimensionalidad y diluye la noción de distancia (maldición de la dimensionalidad). Al seleccionar únicamente `mean radius` y `mean texture`, se redujo el espacio a dos características de baja correlación que capturan simultáneamente la física espacial del tumor y su estructura de cromatina celular.

### 4. ¿Qué ventajas observaste en el agrupamiento jerárquico respecto al otro método?
La principal ventaja analítica del agrupamiento jerárquico es la **ausencia de asumir un $K$ a priori** y la posibilidad de inspeccionar la arquitectura en niveles mediante el dendrograma. Esto permite visualizar el historial topológico completo de fusiones de las muestras a distintas escalas de distancia euclidiana, lo cual es útil en investigación médica para descubrir subtipos histológicos o subclasificaciones multinivel dentro de una misma patología.

### 5. ¿Cómo podrían usarse estos agrupamientos en un contexto médico o de investigación?
En un entorno clínico e investigativo, el clustering no sustituye la biopsia, pero actúa como un motor de descubrimiento biomédico:
* **Descubrimiento de Subtipos Histológicos:** Identificación de variantes atípicas de tumores que no responden a los patrones clásicos de clasificación.
* **Triage y Priorización de Diagnóstico:** Categorización automática de imágenes de biopsia previa inspección patológica para priorizar casos de alto riesgo en la fila de revisión.
* **Sistemas de Soporte a la Decisión Clínica (CDSS):** Servir como capa de preprocesamiento o generación de etiquetas sintéticas antes de entrenar modelos supervisados complejos de pronóstico de supervivencia o de respuesta a quimioterapia.